In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

In [5]:
from google.colab import files

uploaded = files.upload()  # this will open a file picker

KeyboardInterrupt: 

In [4]:
# -----------------------------
# Load Dataset
# -----------------------------
df = pd.read_csv('B:/AI Engineer Intern/Models 10-03-2026/craigslistVehicles.csv')

# Remove bad prices
df = df[(df["price"] > 1000) & (df["price"] < 100000)]

# Drop missing values
df = df.dropna()

# Select useful columns
df = df[[
    "price",
    "year",
    "odometer",
    "manufacturer",
    "fuel",
    "transmission",
    "condition",
    "drive",
    "type"
]]

FileNotFoundError: [Errno 2] No such file or directory: 'B:/AI Engineer Intern/Models 10-03-2026/craigslistVehicles.csv'

In [ ]:
# -----------------------------
# Features and Target
# -----------------------------
X = df.drop("price", axis=1)
y = df["price"]


In [ ]:
# -----------------------------
# Feature Scaling
# -----------------------------
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Scale target
y_scaler = StandardScaler()
y = y_scaler.fit_transform(y.values.reshape(-1,1))

In [ ]:
# -----------------------------
# Train Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42
)

# Convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
# -----------------------------
# Neural Network
# -----------------------------
class CarPriceModel(nn.Module):

    def __init__(self,input_size):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(input_size,128),
            nn.ReLU(),

            nn.Linear(128,64),
            nn.ReLU(),

            nn.Linear(64,32),
            nn.ReLU(),

            nn.Linear(32,1)
        )

    def forward(self,x):
        return self.model(x)

model = CarPriceModel(X_train.shape[1])

In [ ]:
# -----------------------------
# Loss and Optimizer
# -----------------------------
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# -----------------------------
# Training
# -----------------------------
epochs = 200

for epoch in range(epochs):

    model.train()

    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1)%20==0:
        print(f"Epoch {epoch+1}, Loss {loss.item()}")

In [ ]:
# -----------------------------
# Evaluation
# -----------------------------
model.eval()

with torch.no_grad():

    predictions = model(X_test)

# Convert back to numpy
pred = predictions.numpy()
true = y_test.numpy()

# R2 Score
print("R2 Score:", r2_score(true, pred))